In [2]:
# load dataset
import pandas as pd
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

In [3]:
# train test split
from sklearn.model_selection import train_test_split
X = train_df.drop(columns=['price'])
y = train_df['price']
x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
cat_cols = x_train.select_dtypes(include=['object']).columns
num_cols = x_train.select_dtypes(include=['int64','float64']).columns

In [6]:
# Normalization
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

In [7]:
x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed   = preprocessor.transform(x_val)
test_processed    = preprocessor.transform(test_df)

In [8]:
# TensorFlow model bulding
import tensorflow as tf
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(x_train_processed.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)   # regression output
])
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
history = model.fit(
    x_train_processed, y_train,
    validation_data=(x_val_processed, y_val),
    epochs=30,
    batch_size=32
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 10/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan 

In [10]:
model.evaluate(x_val_processed, y_val)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: nan - mae: nan


[nan, nan]

In [11]:
preds = model.predict(test_processed)
pd.DataFrame(preds, columns=['Predicted_Price']).head()

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


,Predicted_Price
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN


In [12]:
model.save("house_price_model.keras")